# Evaluation of IDF of hourly CPM emulators

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import functools
import IPython
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import string
import xarray as xr

from mlde_analysis.idf import calc_spells, calc_pmf, plot_pmf, INTENSITY_BINS, DURATIONS_BINS

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 300

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

## IDF

In [ ]:
%%time

var = "pr"

pmfs = {}

da = TARGET_DAS[var]
spells = []
point_ts_da = da.isel(grid_latitude=[31], grid_longitude=[31], ensemble_member=[0])
# for (lat, lon), point_ts_da in da.groupby(["grid_latitude","grid_longitude"]):
for (sesaon, year), sy_point_ts_da in point_ts_da.groupby(["time.season", "time.year"]):
    spells.extend(calc_spells(sy_point_ts_da.squeeze().values))
pmfs["CPM"] = calc_pmf(spells)

da = PRED_DAS[var]
point_ts_da = da.isel(grid_latitude=[31], grid_longitude=[31], ensemble_member=[0])
for model, model_point_ts_da in point_ts_da.groupby("model"):
    # for (lat, lon), point_ts_da in da.groupby(["grid_latitude","grid_longitude"]):
    # for each season, compute 2D bin count for different durations and intensities
    spells = []
    for (sesaon, year), sy_model_point_ts_da in model_point_ts_da.groupby(["time.season", "time.year"]):
        spells.extend(calc_spells(sy_model_point_ts_da.squeeze().values))
    pmfs[model] = calc_pmf(spells)

In [ ]:
%%time

fig = plt.figure(layout="constrained", figsize=(6, 5))
entries = list(pmfs.keys())
cols = 2
npads = (cols - len(entries) % cols) % cols
grid_spec = np.pad(np.array(entries), (0,npads), mode='constant', constant_values=".").reshape(-1, cols)
axd = fig.subplot_mosaic(grid_spec, sharex=True, sharey=True)

for model, pmf in pmfs.items():
    ax = axd[model]
    shw = plot_pmf(ax, pmf, title=model)

cb = fig.colorbar(
    shw,
    ax=axd.values(),
    location="right",
    pad=0.12,
    shrink=0.8,
    # aspect=40,
    extend="max",
)
cb.set_label("PMF", fontsize="small")
cb.ax.tick_params(labelsize="small")

plt.show()

In [ ]:
client.close()